## NRI Data Check
Access Data from past and current versions (2020,2021,2023,2025)\
Identify and isolate relevant columns, basic exploration, generate summary score for YoY risk index


In [1]:
%pip install arcgis

  Using cached arcgis-2.3.1-cp39-cp39-linux_x86_64.whl
  Using cached pylerc-4.0-py3-none-any.whl
  Using cached geomet-1.1.0-py3-none-any.whl (31 kB)
  Using cached requests_gssapi-1.4.0-py3-none-any.whl (12 kB)
  Using cached puremagic-1.30-py3-none-any.whl (43 kB)
  Using cached python_certifi_win32-1.6.1-py2.py3-none-any.whl (7.3 kB)
  Using cached requests-2.31.0-py3-none-any.whl (62 kB)
  Using cached requests_kerberos-0.15.0-py2.py3-none-any.whl (12 kB)
  Using cached pyspnego-0.12.0-py3-none-any.whl (130 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)
  Using cached keyring-25.7.0-py3-none-any.whl (39 kB)
  Using cached pandas-2.1.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.3 MB)
  Using cached dask-2024.8.0-py3-none-any.whl (1.2 MB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
  Using cached pyshp-3.0.3-py3-none-any.whl (58 kB)
  Using cached pyarrow-21.0.0-cp39-cp39-manylinux_2_28_x86_64.whl (42.7 MB)
  Using c

  Using cached backports.tarfile-1.2.0-py3-none-any.whl (30 kB)
  Using cached setuptools_scm-9.2.2-py3-none-any.whl (62 kB)
  Using cached gssapi-1.11.1.tar.gz (95 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Using cached krb5-0.9.0.tar.gz (236 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Using cached oauthlib-3.3.1-py3-none-any.whl (160 kB)
  Using cached tomli-2.4.0-py3-none-any.whl (14 kB)
  ERROR: Command errored out with exit status 1:
   command: /opt/conda/bin/python /opt/conda/lib/python3.9/site-packages/pip/_vendor/pep517/in_process/_in_process.py build_wheel /tmp/tmpm5fr2grk
       cwd: /tmp/pip-install-wjecy69k/gssapi_11bd560cf3074d3f8ee946287d8beef1
  Complete output (54 lines):
  running bdist_wheel
  running build
  running build_py
  creating build/lib.linux-x86_64-cpython-39/gssapi
  copying gssapi

  ERROR: Command errored out with exit status 1:
   command: /opt/conda/bin/python /opt/conda/lib/python3.9/site-packages/pip/_vendor/pep517/in_process/_in_process.py build_wheel /tmp/tmpl6brzmtu
       cwd: /tmp/pip-install-wjecy69k/krb5_1be5a4bc30e34076ad9c0bea766fc589
  Complete output (82 lines):
  Using krb5-config at 'krb5-config'
  Using /opt/conda/lib/libkrb5.so as Kerberos module for platform checks
  Compiling src/krb5/_ccache.pyx
  Compiling src/krb5/_ccache_mit.pyx
  Compiling src/krb5/_ccache_match.pyx
  Compiling src/krb5/_ccache_support_switch.pyx
  Compiling src/krb5/_cccol.pyx
  Compiling src/krb5/_chpw_message_mit.pyx
  Compiling src/krb5/_context.pyx
  Compiling src/krb5/_context_mit.pyx
  Compiling src/krb5/_creds.pyx
  Skipping src/krb5/_creds_marshal_mit.pyx as it is not supported by the selected Kerberos implementation.
  Compiling src/krb5/_creds_mit.pyx
  Compiling src/krb5/_creds_opt.pyx
  Skipping src/krb5/_creds_opt_heimdal.pyx as it is not supported by the 

In [2]:
import pandas as pd
from arcgis.features import FeatureLayer
import urllib3
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pytest
import seaborn as sns

## Switch to read .csv if needed (saved from previous version on github)

# 1. Suppress SSL Warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.simplefilter("ignore", UserWarning)

def fetch_nri_by_url(service_url, version_label):
    """
    Fetches data directly via REST URL to bypass Item ID retirement issues.
    """
    print(f"Connecting to {version_label}...")
    try:
        # Initialize the layer directly from the URL
        # We set verify_cert=False to stop the SSL noise in restricted environments
        layer = FeatureLayer(service_url)
        
        # Query the data. We use 'out_fields="*"' to get all columns.
        # We also request result_type='standard' to help with the decompression issue.
        query_result = layer.query(where="1=1", out_fields="*", return_geometry=False)
        
        # Convert to DataFrame
        df = query_result.sdf
        
        # Clean up column names (ArcGIS sometimes adds extra prefixes)
        df.columns = [c.upper() for c in df.columns]
        
        print(f"Successfully imported {len(df)} records for {version_label}.")
        return df
    except Exception as e:
        print(f"Failed to fetch {version_label}: {e}")
        return None

# --- REST Service URLs (Updated for 2020-2023 Snapshots) ---
# Note: These URLs point to the persistent FeatureServer endpoints 
# managed by FEMA or the Resilience/Climate Hubs.

urls = {
    "2020_v1_17": "https://services.arcgis.com/XG15cJAlne2vxtgt/arcgis/rest/services/NRI_Counties_v117/FeatureServer/0",
    "2021_v1_18": "https://services.arcgis.com/XG15cJAlne2vxtgt/arcgis/rest/services/NRI_Counties_Prod_v1181_view/FeatureServer/0",
    "2023_v1_19": "https://services.arcgis.com/XG15cJAlne2vxtgt/arcgis/rest/services/National_Risk_Index_Counties_(March_2023)/FeatureServer/0",
    "2025_v1_20": "https://services.arcgis.com/XG15cJAlne2vxtgt/arcgis/rest/services/National_Risk_Index_Counties/FeatureServer/0"
}

# --- Import Data ---
nri_2020 = fetch_nri_by_url(urls["2020_v1_17"], "NRI 2020 (v1.17)")
nri_2021 = fetch_nri_by_url(urls["2021_v1_18"], "NRI 2021 (v1.18)")
nri_2023 = fetch_nri_by_url(urls["2023_v1_19"], "NRI 2023 (v1.19)")
nri_2025 = fetch_nri_by_url(urls["2025_v1_20"], "NRI 2025 (v1.20)")

pd.set_option('display.max_columns', 50)
pd.options.display.float_format = '{:,.2f}'.format

/opt/conda/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


ModuleNotFoundError: No module named 'arcgis'

In [ ]:
print(nri_2020.shape)
nri_2020.iloc[:,0:50].head()

In [ ]:
nri_2020.head(50).columns

In [ ]:
print(nri_2021.shape)
nri_2021.iloc[:,0:22].head()

In [ ]:
print(nri_2023.shape)
nri_2023.iloc[:,0:22].head()

In [ ]:
print(nri_2025.shape)
nri_2025.iloc[:,0:22].head()

# Columns of interest:
State, county, population, 
bldg value, agricultural value, area, 
risk value (dollar amount), risk score (nat'l percentile ranking), risk rating (category), 
expected annual loss value (dollar amount), expected annual loss score (nat'l %), expected annual loss rating (category), 
social vulnerability score (nat'l %), social vulnerability rating (category),
Community resilience score (nat'l %), community resilience rating (category), community resilience value (index from source data), community risk factor

'STATEABBRV', 'COUNTY', 'POPULATION', 'BUILDVALUE', 'AGRIVALUE', 'AREA', 'RISK_VALUE', 'RISK_SCORE', 'RISK_RATING', 'EAL_VALT', 'EAL_SCORE', 'EAL_RATING', 'SOVI_SCORE', 'SOVI_RATING', 'RESL_SCORE', 'RESL_RATING', 'RESL_VALUE', 'CRF_VALUE'

20-21
'RISK_NPCTL', 'EAL_VALT', 'SOVI_VALUE', 'RESL_VALUE'

There are detailed fields for each disaster type, state level percent ranks, exp annual loss rate (dollars as % of bldgs, pop, agri), adjusted exp ann loss rate for social vulnerability and comm resil

# Note on Scores:
Scores in 2020 and 2021 were 1-100 relative index scores (although not national percentiles), where 2023 and 2025 have values that represent dollar amounts (meant to be an absolute indicator), and scores that are nat'l percentile rating 

Using the updated methodology, we can compute retroactive risk values for 2020 and 2021 by replicating their method with sovi_score, resl_score, and eal_valt

In [ ]:
keep_cols_1 = ['NRI_ID',
            'STATEABBRV', 
             'COUNTY', 
             'POPULATION', 
             'BUILDVALUE', 
             'AGRIVALUE', 
             'AREA', 
             'RISK_SCORE', 
             'RISK_RATNG', 
             'RISK_NPCTL', 
             'EAL_SCORE', 
             'EAL_VALT', 
             'EAL_NPCTL', 
             'EAL_RATNG', 
             'SOVI_SCORE', 
             'SOVI_RATNG', 
             'SOVI_VALUE', 
             'RESL_SCORE', 
             'RESL_RATNG', 
             'RESL_VALUE']

keep_cols_2 = ['NRI_ID',
               'STATEABBRV', 
             'COUNTY', 
             'POPULATION', 
             'BUILDVALUE', 
             'AGRIVALUE', 
             'AREA',
             'RISK_VALUE', 
             'RISK_SCORE', 
             'RISK_RATNG', 
             'EAL_SCORE', 
             'EAL_VALT',  
             'EAL_RATNG', 
             'SOVI_SCORE', 
             'SOVI_RATNG',  
             'RESL_SCORE', 
             'RESL_RATNG', 
             'RESL_VALUE',
             'CRF_VALUE']

nri_20 = nri_2020[keep_cols_1]
nri_20.columns = nri_20.columns.str.lower()

nri_21 = nri_2021[keep_cols_1]
nri_21.columns = nri_21.columns.str.lower()

nri_23 = nri_2023[keep_cols_2]
nri_23.columns = nri_23.columns.str.lower()

nri_25 = nri_2025[keep_cols_2]
nri_25.columns = nri_25.columns.str.lower()

print('Shapes:\n'+'='*50)
print('2020', nri_20.shape)
print('2021', nri_21.shape)
print('2023', nri_23.shape)
print('2025', nri_25.shape)

Looks like **89** counties were added in **2023** and **1** more in **2025** for a total of **90**

In [ ]:
# Adding state+county- nri_id ended up being better

nri_20.loc[:, "st_county"] = nri_20["stateabbrv"].str.cat(nri_20["county"], sep='_')
nri_21.loc[:, "st_county"] = nri_21["stateabbrv"].str.cat(nri_21["county"], sep='_')
nri_23.loc[:, "st_county"] = nri_23["stateabbrv"].str.cat(nri_23["county"], sep='_')
nri_25.loc[:, "st_county"] = nri_25["stateabbrv"].str.cat(nri_25["county"], sep='_')

20/21 'risk_score' is the old methodology (unitless, 0-100). 23/25 'risk_score' is national percentile, formerly 'risk_npctl'

## Missing Values- Territories 

In [ ]:
print("\nMissing values total 2020")
print(nri_20.isna().sum().sum())
print("\nMissing values total 2021")
print(nri_21.isna().sum().sum())
print("\nMissing values total 2023")
print(nri_23.isna().sum().sum())
print("\nMissing values total 2025")
print(nri_25.isna().sum().sum())


In [ ]:
nri_23.isna().sum().sort_values(ascending=False)

In [ ]:
nri_25.isna().sum().sort_values(ascending=False)

23 and 25 missing data for 88 rows which appear to be territories

In [ ]:
nri_25[nri_25["crf_value"].isna()].stateabbrv.value_counts()

In [ ]:
nri_25[(nri_25["crf_value"].notna()) & (nri_25["stateabbrv"].isin(["PR","AS","MP","VI","GU"]))]

In [ ]:
nri_25 = nri_25.drop(nri_25[nri_25.stateabbrv.isin(["PR","AS","MP","VI","GU"])].index)
nri_23 = nri_23.drop(nri_23[nri_23.stateabbrv.isin(["PR","AS","MP","VI","GU"])].index)

print('2020', nri_20.shape)
print('2021', nri_21.shape)
print('2023',nri_23.shape)
print('2025',nri_25.shape)

In [ ]:
# Recheck NaNs

print("\nMissing values total 2020")
print(nri_20.isna().sum().sum())
print("\nMissing values total 2021")
print(nri_21.isna().sum().sum())
print("\nMissing values total 2023")
print(nri_23.isna().sum().sum())
print("\nMissing values total 2025")
print(nri_25.isna().sum().sum())

### Investigate different row lengths 

In [ ]:
print("Counties New in 21")
print(len(nri_21[~nri_21["nri_id"].isin(nri_20["nri_id"])]))
print("Counties Removed in 21")
print(len(nri_20[~nri_20["nri_id"].isin(nri_21["nri_id"])]))

In [ ]:
print("Counties New in 23")
print(len(nri_23[~nri_23["nri_id"].isin(nri_21["nri_id"])]))
print("Counties Removed in 23")
print(len(nri_21[~nri_21["nri_id"].isin(nri_23["nri_id"])]))

In [ ]:
# New counties in 23
nri_23[~nri_23["nri_id"].isin(nri_21["nri_id"])]

In [ ]:
# Removed county in 23 (this was dissolved and split into the two separate census areas: https://en.wikipedia.org/wiki/Valdez%E2%80%93Cordova_Census_Area,_Alaska)
nri_21[~nri_21["nri_id"].isin(nri_23["nri_id"])]

In [ ]:
print("Counties New in 25")
print(len(nri_25[~nri_25["nri_id"].isin(nri_23["nri_id"])]))
print("Counties Removed in 25")
print(len(nri_23[~nri_23["nri_id"].isin(nri_25["nri_id"])]))

Connecticut replaced 8 counties with 9 new counties in 2024
Roughly by area (and risk score comparison):
Old = New
- Northwest Hills = Litchfield
- Western Connecticut = Fairfield
- Greater Bridgeport = Fairfield
- South Central = New Haven
- Naugatuck Valley = New Haven (1/3 Litchfield)
- Lower Connecticut River Valley = Middlesex
- Capitol Planning Region = Hartford (1/3 Toland)
- Southeastern Connecticut = New London
- Northeastern Connecticut = Windham

### Addressing County Changes
Since we're interested in looking at risk over time, I propose we add special rules for CT and these AK counties that set the comparison value to the matching old county and for all others compare to themselves. 

So the removed counties won't have 23/25 data, but the new counties will have their 20, 21, 23 data backfilled by the counties they replaced. 

This means our join should end with 2025's 3144 counties, and the removed counties from past years will be represented in the counties that replaced them. 

In [ ]:
nri_25[~nri_25["nri_id"].isin(nri_23["nri_id"])]

In [ ]:
nri_23[~nri_23["nri_id"].isin(nri_25["nri_id"])]

## To do
- Figure out risk value: ✅
  - Divide sovi score/comm resil score. transform to nat'l percentile✅
  - Apply Probability Point Function- triangular distribution with min 0.5, mode 1, max 2✅
  - Recalculate for 25 to ensure correct methodology✅
  - Sanity check against old score✅
- Arrange multi-year df in friendly format
- calculate (or write funtion) for change over time (takes multiple years, difference between max year value and min year)
- Community resilience score methodology change 25 administration- compare counties to 23, anything obvious
- Interesting correlations: how do we interpret or use zillow data to explore the effect of property prices to risk calculation (commercial vs personal??). Almost a normalization based on property values. physical risk vs monetary. Or understand relationship
2/6
- sovi, resil, crf values to Dave to add to regression
- Investigate massive increase in risk values 23, 25
- Investigate top counties for driving overall stats 

## Community Risk Factor Calc

In [ ]:
from scipy import stats

In [ ]:
# Get initial ratio from sovi_score / resl_score
nri_20['comm_risk_ratio_score'] = (
    nri_20['sovi_score'] / nri_20['resl_score']
)

#replicate for 21
nri_21['comm_risk_ratio_score'] = (
    nri_21['sovi_score'] / nri_21['resl_score']
)

print("\n" + "="*70)
print("STEP 1: Community Risk Ratio Calculation")
print("="*70)
print(nri_20[['county', 'sovi_score', 
          'resl_score', 'comm_risk_ratio_score']].head())
print(f"\nRatio range: {nri_20['comm_risk_ratio_score'].min():.3f} to "
      f"{nri_20['comm_risk_ratio_score'].max():.3f}")

In [ ]:
# Convert ratio to national percentile
nri_20['comm_risk_ratio_npctl'] = (
    nri_20['comm_risk_ratio_score'].rank(pct=True, method='average') * 100
)

#replicate for 21
nri_21['comm_risk_ratio_npctl'] = (
    nri_21['comm_risk_ratio_score'].rank(pct=True, method='average') * 100
)

print("\n" + "="*70)
print("STEP 2: National Percentile Ranking")
print("="*70)
print(nri_20[['county', 'comm_risk_ratio_score', 
          'comm_risk_ratio_npctl']].head(10))
print(f"\nPercentile range: {nri_20['comm_risk_ratio_npctl'].min():.1f} to "
      f"{nri_20['comm_risk_ratio_npctl'].max():.1f}")

In [ ]:
# Apply Triangular Distribution PPF (Percent Point Function)
# Input: percentile (0-100), needs to be converted to probability (0-1)

# Define triangular distribution parameters
tri_min = 0.5
tri_mode = 1.0
tri_max = 2.0

# Create the triangular distribution
# scipy.stats.triang requires:
# - c = (mode - min) / (max - min)  # shape parameter
# - loc = min  # location (minimum)
# - scale = max - min  # scale (range)

c = (tri_mode - tri_min) / (tri_max - tri_min)
tri_dist = stats.triang(c=c, loc=tri_min, scale=tri_max - tri_min)

# Apply PPF: convert percentile (0-100) to probability (0-1), then to risk factor
nri_20['comm_risk_factor'] = tri_dist.ppf(
    nri_20['comm_risk_ratio_npctl'] / 100
)

#replicate for 21
nri_21['comm_risk_factor'] = tri_dist.ppf(
    nri_21['comm_risk_ratio_npctl'] / 100
)

print("\n" + "="*70)
print("STEP 3: Community Risk Factor (Triangular PPF)")
print("="*70)
print(nri_20[['county', 'comm_risk_ratio_npctl', 
          'comm_risk_factor']].head(10))
print(f"\nRisk Factor range: {nri_20['comm_risk_factor'].min():.3f} to "
      f"{nri_20['comm_risk_factor'].max():.3f}")

In [ ]:
# Create Risk Value from EAL value * new comm risk factor
nri_20['risk_value'] = nri_20['eal_valt'] * nri_20['comm_risk_factor']

# replicate for 21
nri_21['risk_value'] = nri_21['eal_valt'] * nri_21['comm_risk_factor']

desc20 = nri_20.risk_value.describe()
desc21 = nri_21.risk_value.describe()
desc23 = nri_23.risk_value.describe()
desc25 = nri_25.risk_value.describe()
comp = pd.concat([desc20,desc21,desc23,desc25],axis=1)
comp

Notable increases in the mean, min, median, and max in 23 and 25

In [ ]:
# Check accuracy: compare to 23 and 25 risk_value distributions, recreate risk_npctl and compare. 
nri_20['risk_npctl_check'] = (
    nri_20['risk_value'].rank(pct=True, method='average') * 100
)

print("\n" + "="*70)
print("Check Accuracy: risk_npctl vs new nat'l pct based on value calc")
print("="*70)
print(nri_20[['county', 'risk_npctl', 
          'risk_npctl_check']].head(10))
print("\nExisting dist:")
print(nri_20['risk_npctl'].describe())
print("\n Vs Calculated stats:")
print(nri_20['risk_npctl_check'].describe())

In [ ]:
# values don't match, but dist does, by definition
nri_20['oldvsnew_risk_npctl'] = nri_20['risk_npctl']-nri_20['risk_npctl_check']
nri_20.oldvsnew_risk_npctl.describe()

In [ ]:
# 2020 Check
# check rank order corr, top and bottom 100 agreement, percentile comparisons
from scipy.stats import spearmanr, rankdata

# Rank correlation
spearman_r, _ = spearmanr(nri_20['risk_score'], nri_20['risk_value'])

# Supporting tests
old_top_100 = set(nri_20.nlargest(100, 'risk_score').index)
new_top_100 = set(nri_20.nlargest(100, 'risk_value').index)
top_overlap = len(old_top_100 & new_top_100)

old_pct = rankdata(nri_20['risk_score']) / len(nri_20) * 100
new_pct = rankdata(nri_20['risk_value']) / len(nri_20) * 100
mean_pct_diff = abs(old_pct - new_pct).mean()

# Decision
print(f"Spearman: {spearman_r:.4f} | Top-100: {top_overlap}/100 | Pct Diff: {mean_pct_diff:.2f}")
print("PASS ✅" if spearman_r > 0.90 and mean_pct_diff < 5 else "REVIEW ⚠️")

In [ ]:
# 2021 check
# Rank correlation
spearman_r, _ = spearmanr(nri_21['risk_score'], nri_21['risk_value'])

# Supporting tests
old_top_100 = set(nri_21.nlargest(100, 'risk_score').index)
new_top_100 = set(nri_21.nlargest(100, 'risk_value').index)
top_overlap = len(old_top_100 & new_top_100)

old_pct = rankdata(nri_21['risk_score']) / len(nri_21) * 100
new_pct = rankdata(nri_21['risk_value']) / len(nri_21) * 100
mean_pct_diff = abs(old_pct - new_pct).mean()

# Decision
print(f"Spearman: {spearman_r:.4f} | Top-100: {top_overlap}/100 | Pct Diff: {mean_pct_diff:.2f}")
print("PASS ✅" if spearman_r > 0.90 and mean_pct_diff < 5 else "REVIEW ⚠️")

## Risk Index Recreation Test: CRF Value

In [ ]:
# Recreating for 25, using sovi score, bc no sovi value present, and docs mention score
nri_25['comm_risk_ratio_value'] = (
    nri_25['sovi_score'] / nri_25['resl_score']
)

print("\n" + "="*70)
print("STEP 1: Community Risk Ratio Calculation")
print("="*70)
print(nri_25[['county', 'sovi_score', 
          'resl_score', 'comm_risk_ratio_value']].head())
print(f"\nRatio range: {nri_25['comm_risk_ratio_value'].min():.3f} to "
      f"{nri_25['comm_risk_ratio_value'].max():.3f}")

In [ ]:
nri_25['comm_risk_ratio_npctl'] = (
    nri_25['comm_risk_ratio_value'].rank(pct=True, method='average') * 100
)

print("\n" + "="*70)
print("STEP 2: National Percentile Ranking")
print("="*70)
print(nri_25[['county', 'comm_risk_ratio_value', 
          'comm_risk_ratio_npctl']].head(10))
print(f"\nPercentile range: {nri_25['comm_risk_ratio_npctl'].min():.1f} to "
      f"{nri_25['comm_risk_ratio_npctl'].max():.1f}")

In [ ]:
# STEP 3: Apply Triangular Distribution PPF
# ============================================================================
# PPF (Percent Point Function) = Inverse CDF
# For triangular distribution: min=0.5, mode=1, max=2
# Input: percentile (0-100), needs to be converted to probability (0-1)

# Define triangular distribution parameters
tri_min = 0.5
tri_mode = 1.0
tri_max = 2.0

# Create the triangular distribution
# scipy.stats.triang requires:
# - c = (mode - min) / (max - min)  # shape parameter
# - loc = min  # location (minimum)
# - scale = max - min  # scale (range)

c = (tri_mode - tri_min) / (tri_max - tri_min)
tri_dist = stats.triang(c=c, loc=tri_min, scale=tri_max - tri_min)

# Apply PPF: convert percentile (0-100) to probability (0-1), then to risk factor
nri_25['comm_risk_factor'] = tri_dist.ppf(
    nri_25['comm_risk_ratio_npctl'] / 100
)

print("\n" + "="*70)
print("STEP 3: Community Risk Factor (Triangular PPF)")
print("="*70)
print(nri_25[['county', 'comm_risk_ratio_npctl', 
          'comm_risk_factor']].head(10))
print(f"\nRisk Factor range: {nri_25['comm_risk_factor'].min():.3f} to "
      f"{nri_25['comm_risk_factor'].max():.3f}")

In [ ]:
new_crf = nri_25.comm_risk_factor.describe()
crf = nri_25.crf_value.describe()
comp = pd.concat([new_crf,crf],axis=1)
comp

### This confirms the correct method is to use sovi_score and resl_score (as shown in the 25 data- these are 0-100 values)

In [ ]:
# Clean up dfs and created columns
nri_20.drop(columns=['comm_risk_ratio_score','comm_risk_ratio_npctl','risk_npctl_check','oldvsnew_risk_npctl'],inplace=True)
nri_20.rename(columns={'comm_risk_factor':'crf_value'},inplace=True)

nri_25.drop(columns=['comm_risk_ratio_value','comm_risk_ratio_npctl','comm_risk_factor'],inplace=True)

nri_21.drop(columns=['comm_risk_ratio_score','comm_risk_ratio_npctl'],inplace=True)
nri_21.rename(columns={'comm_risk_factor':'crf_value'},inplace=True)

## Risk Value Retroactive Calculation Recap
Successful recreation of the 2025 CRF value shows we are applying the method correctly. 2020 and 21 risk values are predictably different from the risk score used at the time due to the change in methodology, but they are relatively similar as shown by the spearman rank correlation of close to 0.9

## Set up Risk df for YoY comparison

In [ ]:
# Select risk and id columns
risk_25 = nri_25[['nri_id', 'stateabbrv', 'county','risk_value']].copy()
risk_23 = nri_23[['nri_id', 'risk_value']].copy()
risk_21 = nri_21[['nri_id', 'risk_value']].copy()
risk_20 = nri_20[['nri_id', 'risk_value']].copy()

# rename columns
risk_25.rename(columns={'risk_value': 'risk_val_2025'}, inplace=True)
risk_23.rename(columns={'risk_value': 'risk_val_2023'}, inplace=True)
risk_21.rename(columns={'risk_value': 'risk_val_2021'}, inplace=True)
risk_20.rename(columns={'risk_value': 'risk_val_2020'}, inplace=True)

risk_df = (risk_25
            .merge(risk_23, on='nri_id', how='outer')
            .merge(risk_21, on='nri_id', how='outer')
            .merge(risk_20, on='nri_id', how='outer'))

# add CT and AK county changes to fill in missing data and drop old counties
risk_df.shape

In [ ]:
risk_df.head(10)

In [ ]:
# Backfill data for special cases and delete removed county rows


In [ ]:
# Calculate total risk for each year
total_risk = {
    2020: risk_df['risk_val_2020'].sum(),
    2021: risk_df['risk_val_2021'].sum(),
    2023: risk_df['risk_val_2023'].sum(),
    2025: risk_df['risk_val_2025'].sum()
}

# Convert to lists for plotting
years = list(total_risk.keys())
total_risk = list(total_risk.values())

plt.figure(figsize=(12, 7))

# Plot the line
plt.plot(years, total_risk, 
         marker='o',           # Circle markers at each point
         markersize=12,        # Size of markers
         linewidth=3,          # Thickness of line
         color='#2E86AB',      # Nice blue color
         markerfacecolor='#A23B72',  # Different color for markers
         markeredgewidth=2,
         markeredgecolor='white')

# Add value labels on each point
for year, value in zip(years, total_risk):
    plt.text(year, value, 
             f'${value/1e9:.2f}B',  # Format as billions
             ha='center', 
             va='bottom',
             fontsize=11,
             fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8))

# Styling
plt.title('Total Risk Value Across All U.S. Counties (2020-2025)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Year', fontsize=13, fontweight='bold')
plt.ylabel('Total Risk Value (USD)', fontsize=13, fontweight='bold')

# Format y-axis to show billions
ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e9:.1f}B'))

# Add grid for readability
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.7)

# Set x-axis to only show our years
plt.xticks(years, fontsize=11)
plt.yticks(fontsize=11)

# Add some padding to y-axis
y_range = max(total_risk) - min(total_risk)
plt.ylim(min(total_risk) - y_range*0.1, max(total_risk) + y_range*0.15)

# Tight layout
plt.tight_layout()

plt.show()

In [ ]:
avg_risk_by_year = {
    2020: risk_df['risk_val_2020'].mean(),
    2021: risk_df['risk_val_2021'].mean(),
    2023: risk_df['risk_val_2023'].mean(),
    2025: risk_df['risk_val_2025'].mean()
}

years_avg = list(avg_risk_by_year.keys())
avg_risk = list(avg_risk_by_year.values())

plt.figure(figsize=(12, 7))
plt.plot(years_avg, avg_risk, 
         marker='o',
         markersize=12,
         linewidth=3,
         color='#F18F01',      # Orange color
         markerfacecolor='#C73E1D',
         markeredgewidth=2,
         markeredgecolor='white')

# Add value labels
for year, value in zip(years_avg, avg_risk):
    plt.text(year, value, 
             f'${value/1e6:.2f}M',  # Format as millions
             ha='center', 
             va='bottom',
             fontsize=11,
             fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8))

plt.title('Average Risk Value Per County (2020-2025)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Year', fontsize=13, fontweight='bold')
plt.ylabel('Average Risk Value (USD)', fontsize=13, fontweight='bold')

# Format y-axis to show millions
ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.7)
plt.xticks(years_avg, fontsize=11)
plt.yticks(fontsize=11)

y_range = max(avg_risk) - min(avg_risk)
plt.ylim(min(avg_risk) - y_range*0.1, max(avg_risk) + y_range*0.15)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pytest
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)
ymin,ymax = ax.get_ylim() 

ax.hist(nri_25.risk_value, bins=1000)
ax.vlines(x=nri_25.risk_value.mean(),ymin=ymin,ymax=ymax,colors='r',linestyles='--',label='Avg')
plt.show()

In [ ]:
# pd.options.display.float_format = '{:,.2f}'.format
#nri_25.risk_value = nri_25.risk_value.round(2)
nri_25.style.format({"risk_value": "{:,}"})
nri_25[nri_25['risk_value'] > 1000000000].sort_values('risk_value',ascending=False)

# nri_25['risk_value' > 1000000000]


In [ ]:
bil_25 = nri_25[nri_25['risk_value'] > 1000000000].sort_values('risk_value',ascending=False)
bil_25['county_state'] = bil_25.county + ', ' + bil_25.stateabbrv
# bil_25.risk_value.plot(kind='bar')

plt.figure(figsize=(12, 4), dpi=150)
plt.bar(bil_25['county_state'],bil_25.risk_value, linewidth=2)
plt.xticks(rotation=45,ha='right')
plt.title("Billion Dollar Risk Values")
plt.ylabel("Risk Value", fontsize=12)
plt.xlabel("State and County", fontsize=12)
plt.show()

In [ ]:
import numpy as np

correlation_matrix = nri_25.corr(method='spearman',numeric_only=True) # numeric_only=True to handle non-numeric columns
correlation_matrix

mask = np.ones_like(correlation_matrix, dtype=bool)
mask = np.triu(mask)
plt.figure(figsize=(8, 6)) # Adjust figure size as needed
sns.heatmap(correlation_matrix, annot=True, cmap='Blues', fmt=".2f", linewidths=.5, mask=mask)
plt.title('NRI Heatmap')
plt.show()

To Dave: County by year with risk, resilience, eal, sovi
- what goes into resilience value/score, sovi value/score (census questions- look at technical doc), risk factor separately
- Sovi: score only nat'l percentile
- Resil: index factor
- Community Resil Factor: sovi/resil 

In [ ]:
sovi = nri_25.sovi_score.describe()
resl = nri_25.resl_value.describe()
resl2 = nri_25.resl_score.describe()
crf = nri_25.crf_value.describe()
resil = pd.concat([sovi,resl2,resl,crf],axis=1)
resil

In [ ]:
sovi = nri_20.sovi_score.describe()
resl = nri_20.resl_value.describe()
crf = nri_20.crf_value.describe()
resil = pd.concat([sovi,resl,crf],axis=1)
resil

In [ ]:
nri_25.head()

In [ ]:
# Select columns, create year column
core_25 = nri_25[['nri_id', 'stateabbrv', 'county','risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()
core_25['year'] = 2025
core_23 = nri_23[['nri_id', 'risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()
core_23['year'] = 2023
core_21 = nri_21[['nri_id', 'risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()
core_21['year'] = 2021
core_20 = nri_20[['nri_id', 'risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()
core_20['year'] = 2020

core_df_long = pd.concat([core_20, core_21, core_23, core_25], 
                         ignore_index=True)
core_df_long = core_df_long.sort_values(['stateabbrv', 'county', 'year']).reset_index(drop=True)

print(core_df_long.shape)
core_df_long.head()

In [ ]:
# Select columns
corew_25 = nri_25[['nri_id', 'stateabbrv', 'county','risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()
corew_23 = nri_23[['nri_id', 'risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()
corew_21 = nri_21[['nri_id', 'risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()
corew_20 = nri_20[['nri_id', 'risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']].copy()

# rename columns
corew_25.rename(columns={'risk_value': 'risk_val_2025',
                        'eal_valt': 'eal_valt_2025',
                        'sovi_score': 'sovi_score_2025',
                        'resl_score': 'resl_score_2025',
                        'crf_value': 'crf_value_2025'}, inplace=True)
corew_23.rename(columns={'risk_value': 'risk_val_2023',
                        'eal_valt': 'eal_valt_2023',
                        'sovi_score': 'sovi_score_2023',
                        'resl_score': 'resl_score_2023',
                        'crf_value': 'crf_value_2023'}, inplace=True)
corew_21.rename(columns={'risk_value': 'risk_val_2021',
                        'eal_valt': 'eal_valt_2021',
                        'sovi_score': 'sovi_score_2021',
                        'resl_score': 'resl_score_2021',
                        'crf_value': 'crf_value_2021'}, inplace=True)
corew_20.rename(columns={'risk_value': 'risk_val_2020',
                        'eal_valt': 'eal_valt_2020',
                        'sovi_score': 'sovi_score_2020',
                        'resl_score': 'resl_score_2020',
                        'crf_value': 'crf_value_2020'}, inplace=True)

core_df_wide = (corew_25
            .merge(corew_23, on='nri_id', how='outer')
            .merge(corew_21, on='nri_id', how='outer')
            .merge(corew_20, on='nri_id', how='outer'))

# add CT and AK county changes to fill in missing data and drop old counties
print(core_df_wide.shape)
core_df_wide.head()

In [ ]:
core_df_long.to_csv("NRI_Long.csv", index=True)
core_df_wide.to_csv("NRI_Wide.csv", index=True)